In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql.functions import *

In [0]:
def write_to_silver(input_df,target_table,merge_condn,columns_to_update):
    
    
    final_df = (
                    input_df
                    .withColumn('created_timestamp',current_timestamp())
                    .withColumn('updated_timestamp',current_timestamp())
                    )
    
    if not spark.catalog.tableExists(target_table):
        final_df.write.mode("overwrite").format("delta").saveAsTable(target_table)

    else:
        delta_table = DeltaTable.forName(spark, target_table)  
        update_map={f"t.{column}":f"s.{column}" for column in columns_to_update}
        update_map["t.updated_timestamp"]="s.updated_timestamp"

        (
            delta_table.alias('t')
            .merge(final_df.alias('s'), merge_condn)
            .whenMatchedUpdate(
                condition="s.batch_id>=t.batch_id",
                set=update_map

            )
            .whenNotMatchedInsertAll()
            .execute()
        )
